# Label-Free Accuracy Estimation for Tabular Classifiers Under Natural Distribution Shift
CPU runtime is enough (Runtime > Change runtime type > CPU). Expected time: roughly 40-90 minutes.

**Question:** when a trained tabular model meets shifted, *unlabeled* data, which estimator best predicts its accuracy?

**Data:** Folktables / ACS (US Census Bureau public-use microdata; Ding et al., 2021), downloaded automatically. Three tasks: income, public coverage, employment.
**Shifts:** geographic (CA 2018 -> 8 states), temporal (CA 2014 -> CA 2015-2018), and combined (CA 2014 -> 4 states in 2018).
**Models:** LogReg, RandomForest, MLP, XGBoost, LightGBM, CatBoost (default hyperparameters).
**Estimators (all label-free on the target):**
- `naive` source validation accuracy (no-shift assumption)
- `AC` average confidence; `AC_TS` average confidence after temperature scaling
- `DoC` difference of confidences (Guillory et al., 2021)
- `ATC` average thresholded confidence (Garg et al., 2022)
- `AGREE` agreement between two independently trained copies (Jiang et al., 2022); `AGREE_DoC` agreement used as a DoC-style drop
- `LEARNED` linear model on label-free shift features, evaluated leave-one-task-out (needs labeled shifted data from *other* tasks; flagged as a different label-access assumption)

Upload `results.csv` (and `results_bundle.zip`) back to the chat when done.

In [ ]:
!pip -q install folktables xgboost lightgbm catboost

In [ ]:
import os, time, warnings, platform, sys
import numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, log_loss
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')

SEEDS = [0, 1, 2, 3, 4]
N_TRAIN, N_VAL, N_TEST = 10000, 3000, 5000
TASKS = ['ACSIncome', 'ACSPublicCoverage', 'ACSEmployment']
MODELS = ['LogReg', 'RandForest', 'MLP', 'XGBoost', 'LightGBM', 'CatBoost']
STATES = ['TX', 'NY', 'FL', 'IL', 'PA', 'OH', 'MS', 'WA']
SCENARIOS = [
    dict(name='geo',         source=('CA', '2018'), targets=[(s, '2018') for s in STATES]),
    dict(name='temporal',    source=('CA', '2014'), targets=[('CA', y) for y in ['2015', '2016', '2017', '2018']]),
    dict(name='geotemporal', source=('CA', '2014'), targets=[(s, '2018') for s in ['TX', 'NY', 'MS', 'WA']]),
]
print(platform.platform(), sys.version.split()[0])

## Models, data, estimators

In [ ]:
def make_model(name, seed):
    imp = lambda: SimpleImputer(strategy='median')
    return {
        'LogReg':     lambda: make_pipeline(imp(), StandardScaler(), LogisticRegression(max_iter=1000)),
        'RandForest': lambda: make_pipeline(imp(), RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=seed)),
        'MLP':        lambda: make_pipeline(imp(), StandardScaler(), MLPClassifier((128, 64), early_stopping=True, max_iter=200, random_state=seed)),
        'XGBoost':    lambda: XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, tree_method='hist', random_state=seed, n_jobs=-1),
        'LightGBM':   lambda: LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=seed, verbose=-1),
        'CatBoost':   lambda: CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1, random_seed=seed, verbose=0, thread_count=-1),
    }[name]()

def predict_p1(model, X): return model.predict_proba(X)[:, 1]

def logit(p):
    p = np.clip(p, 1e-6, 1-1e-6); return np.log(p/(1-p))
def fit_temp(p, y):
    z = logit(p); f = lambda T: log_loss(y, 1/(1+np.exp(-z/T)), labels=[0, 1])
    return minimize_scalar(f, bounds=(0.05, 20), method='bounded').x
def apply_temp(p, T): return 1/(1+np.exp(-logit(p)/T))

def shift_auc(Xs, Xt):
    # domain-classifier AUC: 0.5 = no covariate shift, 1.0 = fully separable (label-free)
    n = min(len(Xs), len(Xt), 3000)
    Xa = np.vstack([Xs[:n], Xt[:n]]); ya = np.r_[np.zeros(n), np.ones(n)]
    p = cross_val_predict(LGBMClassifier(n_estimators=100, verbose=-1), Xa, ya, cv=3, method='predict_proba')[:, 1]
    return roc_auc_score(ya, p)

def estimators(pv, yv, qv, pt, qt):
    # pv,qv: source-validation probabilities from model A and B; pt,qt: same on UNLABELED target.
    conf = lambda p: np.maximum(p, 1-p)
    cv, ct = conf(pv), conf(pt)
    acc_v = ((pv >= .5) == yv).mean()
    T = fit_temp(pv, yv); ctT = conf(apply_temp(pt, T))
    thr = np.quantile(cv, 1 - acc_v)                        # ATC: fraction of val below thr equals val error
    agree_v = ((pv >= .5) == (qv >= .5)).mean(); agree_t = ((pt >= .5) == (qt >= .5)).mean()
    est = dict(est_naive=acc_v, est_AC=ct.mean(), est_AC_TS=ctT.mean(), est_DoC=acc_v - (cv.mean() - ct.mean()),
               est_ATC=(ct >= thr).mean(), est_AGREE=agree_t, est_AGREE_DoC=acc_v - (agree_v - agree_t))
    est = {k: float(np.clip(v, 0, 1)) for k, v in est.items()}
    est.update(acc_val=acc_v, conf_drop=cv.mean() - ct.mean(), agree_drop=agree_v - agree_t)
    return est

from folktables import ACSDataSource, ACSIncome, ACSPublicCoverage, ACSEmployment
TASK_OBJ = {'ACSIncome': ACSIncome, 'ACSPublicCoverage': ACSPublicCoverage, 'ACSEmployment': ACSEmployment}
_cache = {}
def _load_state_year(state, year):
    df = ACSDataSource(survey_year=year, horizon='1-Year', survey='person', root_dir='acs_data').get_data(states=[state], download=True)
    for tn, T in TASK_OBJ.items():
        X, y, _ = T.df_to_numpy(df); _cache[(tn, state, year)] = (X.astype(np.float32), y.astype(int))
    del df
def load_acs(task, state, year):
    if (task, state, year) not in _cache: _load_state_year(state, year)
    return _cache[(task, state, year)]

## One experimental unit = (task, scenario, seed)

In [ ]:
def run_one(task, seed, scen, get_data, skip=()):
    rng = np.random.RandomState(seed)
    Xs, ys = get_data(task, *scen['source'])
    perm = rng.permutation(len(Xs)); a, b = N_TRAIN, N_TRAIN + N_VAL
    Xtr, ytr = Xs[perm[:a]], ys[perm[:a]]; Xv, yv = Xs[perm[a:b]], ys[perm[a:b]]
    domains = {'ID': (Xs[perm[b:b+N_TEST]], ys[perm[b:b+N_TEST]])}; meta = {'ID': (0.5, 0.0)}
    for t in scen['targets']:
        Xt, yt = get_data(task, *t); j = rng.permutation(len(Xt))[:N_TEST]
        name = f'{t[0]}-{t[1]}'; domains[name] = (Xt[j], yt[j])
        meta[name] = (shift_auc(Xtr, Xt[j]), float(yt[j].mean() - ytr.mean()))
    boot = rng.choice(len(Xtr), len(Xtr))
    out = []
    for mname in MODELS:
        if mname in skip: continue
        try:
            t0 = time.time()
            A = make_model(mname, seed); A.fit(Xtr, ytr)
            B = make_model(mname, seed + 1000); B.fit(Xtr[boot], ytr[boot])      # second copy: other seed + bootstrap sample
            ft = time.time() - t0
            pv, qv = predict_p1(A, Xv), predict_p1(B, Xv)
            for d, (X, y) in domains.items():
                pt, qt = predict_p1(A, X), predict_p1(B, X)
                r = estimators(pv, yv, qv, pt, qt)
                r.update(task=task, scenario=scen['name'], seed=seed, domain=d, shifted=int(d != 'ID'), model=mname,
                         true_acc=float(((pt >= .5) == y).mean()), shift_auc=meta[d][0], prev_shift=meta[d][1], fit_s=ft)
                out.append(r)
        except Exception as e:
            print('FAILED', mname, repr(e))
    return out

## Main loop (saves after every unit; safe to re-run: only missing model rows are computed)

In [ ]:
RES = 'results.csv'
rows, have_all = [], set()
if os.path.exists(RES):
    old = pd.read_csv(RES); rows = old.to_dict('records'); have_all = set(zip(old.task, old.scenario, old.seed, old.model))
wanted = set(MODELS)
for task in TASKS:
    for scen in SCENARIOS:
        for seed in SEEDS:
            have = {m for (t, s_, sd, m) in have_all if (t, s_, sd) == (task, scen['name'], seed)}
            if wanted <= have: continue
            t0 = time.time(); print(f'>> {task} | {scen["name"]} | seed {seed}')
            rows += run_one(task, seed, scen, load_acs, skip=have)
            pd.DataFrame(rows).to_csv(RES, index=False); print(f'   done in {time.time()-t0:.0f}s')
print(pd.read_csv(RES).shape)

## Analysis: tables, statistical tests, figures

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon, pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
os.makedirs('figs', exist_ok=True)
df = pd.read_csv(RES)
EST = ['est_naive', 'est_AC', 'est_AC_TS', 'est_DoC', 'est_ATC', 'est_AGREE', 'est_AGREE_DoC']

# Learned label-free estimator, leave-one-task-out (features available without target labels)
df['true_drop'] = df.acc_val - df.true_acc; feats = ['shift_auc', 'conf_drop', 'agree_drop']; df['est_LEARNED'] = np.nan
for t in df.task.unique():
    tr, te = df[df.task != t], df.task == t
    df.loc[te, 'est_LEARNED'] = (df.loc[te, 'acc_val'] - LinearRegression().fit(tr[feats], tr.true_drop).predict(df.loc[te, feats])).clip(0, 1)
EST.append('est_LEARNED')
for e in EST:
    df['err_' + e] = (df[e] - df.true_acc) * 100; df['abs_' + e] = df['err_' + e].abs()
S = df[df.shifted == 1].copy(); I = df[df.shifted == 0]

# Table 1: error of each estimator on shifted data (percentage points) + ID reference
t1 = pd.DataFrame({e: dict(MAE_shifted=S['abs_'+e].mean(), RMSE_shifted=np.sqrt((S['err_'+e]**2).mean()), bias=S['err_'+e].mean(),
                           P90_abs_err=S['abs_'+e].quantile(.9), pct_within_5pp=(S['abs_'+e] <= 5).mean()*100, MAE_ID=I['abs_'+e].mean()) for e in EST}).T.round(3)
t1.to_csv('table1_estimator_error.csv'); display(t1.sort_values('MAE_shifted'))

# Table 2: MAE by model x estimator (shifted)
t2 = S.groupby('model')[['abs_' + e for e in EST]].mean().round(3); t2.columns = EST; t2.to_csv('table2_mae_by_model.csv'); display(t2)

# Table 3: MAE by shift severity tercile
S['severity'] = pd.qcut(S.shift_auc, 3, labels=['low', 'mid', 'high'])
t3 = S.groupby('severity')[['abs_' + e for e in EST]].mean().round(3); t3.columns = EST; t3.to_csv('table3_mae_by_severity.csv'); display(t3)

# Table 4: how well do estimates track true accuracy across domains (means over seeds)
G = S.groupby(['task', 'scenario', 'domain', 'model'])[EST + ['true_acc']].mean().reset_index()
t4 = pd.DataFrame({e: dict(pearson_r=pearsonr(G[e], G.true_acc)[0], spearman_rho=spearmanr(G[e], G.true_acc)[0],
                           R2=1 - ((G.true_acc - G[e])**2).sum() / ((G.true_acc - G.true_acc.mean())**2).sum()) for e in EST}).T.round(3)
t4.to_csv('table4_tracking.csv'); display(t4)

# Statistical tests on absolute error (blocks = task/scenario/seed/domain/model)
piv = S.pivot_table(index=['task', 'scenario', 'seed', 'domain', 'model'], values=['abs_' + e for e in EST]).dropna()
print('Friedman:', friedmanchisquare(*[piv['abs_' + e] for e in EST]))
pv = [(a, b, wilcoxon(piv['abs_' + a], piv['abs_' + b]).pvalue) for i, a in enumerate(EST) for b in EST[i+1:]]
pw = pd.DataFrame(pv, columns=['a', 'b', 'p']).sort_values('p'); m = len(pw)
pw['p_holm'] = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * pw.p.values))
pw.to_csv('pairwise_wilcoxon_holm.csv', index=False); display(pw.head(15))

# Figures
lab = lambda e: e.replace('est_', '')
order = t1.sort_values('MAE_shifted').index
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.bar([lab(e) for e in order], t1.loc[order, 'MAE_shifted'])
ax.set_ylabel('MAE of accuracy estimate (pp), shifted data'); ax.set_title('Estimator error'); plt.xticks(rotation=30); plt.tight_layout(); plt.savefig('figs/fig1_mae.png', dpi=300); plt.show()

show = ['est_naive', 'est_AC', 'est_DoC', 'est_ATC', 'est_AGREE_DoC', 'est_LEARNED']
fig, axs = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for a, e in zip(axs.ravel(), show):
    a.scatter(G.true_acc, G[e], s=8, alpha=.5); lo, hi = G.true_acc.min() - .02, G.true_acc.max() + .02
    a.plot([lo, hi], [lo, hi], 'k--', lw=.8); a.set_title(lab(e)); a.set_xlabel('True accuracy'); a.set_ylabel('Estimated accuracy')
plt.tight_layout(); plt.savefig('figs/fig2_est_vs_true.png', dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
for e in EST: ax.plot(t3.index.astype(str), t3[e], marker='o', label=lab(e))
ax.set_xlabel('Shift severity tercile (domain-classifier AUC)'); ax.set_ylabel('MAE (pp)'); ax.legend(fontsize=7); plt.tight_layout(); plt.savefig('figs/fig3_mae_vs_severity.png', dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(8, 4)); im = ax.imshow(t2.values, aspect='auto'); ax.set_xticks(range(len(EST))); ax.set_xticklabels([lab(e) for e in EST], rotation=30)
ax.set_yticks(range(len(t2))); ax.set_yticklabels(t2.index)
for i in range(t2.shape[0]):
    for j in range(t2.shape[1]): ax.text(j, i, f'{t2.values[i, j]:.1f}', ha='center', va='center', fontsize=7, color='w')
plt.colorbar(im, label='MAE (pp)'); plt.tight_layout(); plt.savefig('figs/fig4_mae_heatmap.png', dpi=300); plt.show()

df.to_csv('results_with_learned.csv', index=False)
!zip -qr results_bundle.zip results.csv results_with_learned.csv table*.csv pairwise_wilcoxon_holm.csv figs
from google.colab import files; files.download('results_bundle.zip')